# FX Fintech Product Analytics

This notebook analyzes synthetic product behavior and historical Yahoo Finance FX data.
Product results are illustrative and do not represent actual company performance.

# 0. Project Setup

## 0.1 Import Libraries

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import pickle
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

## 0.2 Project Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "fx_fintech_product_analytics.db"
SQL_DIR = PROJECT_ROOT / "sql"
CSV_DIR = PROJECT_ROOT / "outputs" / "csv"
MODEL_DIR = PROJECT_ROOT / "models"
FX_CACHE_DIR = PROJECT_ROOT / "data" / "external"
FX_CACHE_PATH = FX_CACHE_DIR / "yahoo_fx_daily.csv"
FX_METADATA_PATH = FX_CACHE_DIR / "yahoo_fx_metadata.json"

CSV_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FX_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FX_START_DATE = "2016-01-01"
FX_END_DATE = "2026-07-10"  # Yahoo Finance end date is exclusive
FX_TICKERS = {
    "USDKRW": "USDKRW=X",
    "JPYKRW": "JPYKRW=X",
    "EURKRW": "EURKRW=X",
    "SGDKRW": "SGDKRW=X"
}

print("Database: data/fx_fintech_product_analytics.db")
print("CSV output: outputs/csv/")
print("Random state:", RANDOM_STATE)

In [ ]:
# Build the clean event view and materialized user lifecycle mart
conn = sqlite3.connect(DB_PATH)

build_mart_sql = (SQL_DIR / "01_build_data_mart.sql").read_text(encoding="utf-8")
conn.executescript(build_mart_sql)

print("Data mart ready")

# 1. Product Analytics

## 1.1 Dataset Overview

In [ ]:
table_names = [
    "users",
    "events",
    "transactions",
    "support_tickets",
    "marketing_spend",
    "fx_rates_hourly",
    "modeling_user_month_snapshots"
]

table_overview = []

for table_name in table_names:
    row_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS rows FROM {table_name}",
        conn
    ).loc[0, "rows"]

    table_overview.append({
        "table": table_name,
        "rows": row_count
    })

table_overview = pd.DataFrame(table_overview)
table_overview

In [ ]:
# Check date coverage for the main product tables
date_coverage_query = '''
SELECT
    'users' AS table_name,
    MIN(signup_timestamp) AS min_date,
    MAX(signup_timestamp) AS max_date
FROM users

UNION ALL

SELECT
    'transactions',
    MIN(transaction_timestamp),
    MAX(transaction_timestamp)
FROM transactions

UNION ALL

SELECT
    'modeling_user_month_snapshots',
    MIN(observation_date),
    MAX(observation_date)
FROM modeling_user_month_snapshots
'''

date_coverage = pd.read_sql_query(date_coverage_query, conn)
date_coverage

The SQLite product tables are synthetic. The legacy `fx_rates_hourly` table is simulated and is only used for descriptive regime analysis. The volatility prediction model later in this notebook uses Yahoo Finance data.

## 1.2 Data Quality Checks

### 1.2.1 Missing Values

In [ ]:
quality_tables = [
    "users",
    "events",
    "transactions",
    "support_tickets",
    "marketing_spend",
    "modeling_user_month_snapshots"
]

missing_value_records = []

for table_name in quality_tables:
    table_columns = pd.read_sql_query(
        f"PRAGMA table_info({table_name})",
        conn
    )["name"].tolist()

    for column_name in table_columns:
        missing_count = pd.read_sql_query(
            f'''
            SELECT COUNT(*) AS missing_count
            FROM {table_name}
            WHERE "{column_name}" IS NULL
               OR "{column_name}" = ''
            ''',
            conn
        ).loc[0, "missing_count"]

        if missing_count > 0:
            missing_value_records.append({
                "check": "missing_values",
                "table": table_name,
                "column": column_name,
                "issue_count": missing_count,
                "severity": "warning",
                "detail": "Review whether the field is conditionally optional."
            })

missing_values = pd.DataFrame(missing_value_records)
missing_values

### 1.2.2 Duplicate Data

In [ ]:
duplicate_user_count = pd.read_sql_query(
    "SELECT COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_count FROM users",
    conn
).loc[0, "duplicate_count"]

duplicate_transaction_count = pd.read_sql_query(
    "SELECT COUNT(*) - COUNT(DISTINCT transaction_id) AS duplicate_count FROM transactions",
    conn
).loc[0, "duplicate_count"]

# Event duplicates use the business event key rather than event_id
event_keys = pd.read_sql_query(
    '''
    SELECT user_id, session_id, event_timestamp, event_name
    FROM events
    ''',
    conn
)
duplicate_event_count = event_keys.duplicated(keep="first").sum()
del event_keys

duplicate_checks = pd.DataFrame([
    {
        "check": "duplicate_user_id",
        "table": "users",
        "column": "user_id",
        "issue_count": duplicate_user_count,
        "severity": "error",
        "detail": "Each user_id should identify one user."
    },
    {
        "check": "duplicate_transaction_id",
        "table": "transactions",
        "column": "transaction_id",
        "issue_count": duplicate_transaction_count,
        "severity": "error",
        "detail": "Each transaction_id should identify one transaction."
    },
    {
        "check": "duplicate_event_rows",
        "table": "events",
        "column": "user_id, session_id, event_timestamp, event_name",
        "issue_count": duplicate_event_count,
        "severity": "warning",
        "detail": "clean_events keeps one row for each duplicated event key."
    }
])

duplicate_checks

### 1.2.3 Date and Transaction Validity

In [ ]:
validation_queries = {
    "invalid_signup_date": '''
        SELECT COUNT(*) AS issue_count
        FROM users
        WHERE signup_timestamp IS NOT NULL
          AND julianday(signup_timestamp) IS NULL
    ''',
    "invalid_event_date": '''
        SELECT COUNT(*) AS issue_count
        FROM events
        WHERE event_timestamp IS NOT NULL
          AND julianday(event_timestamp) IS NULL
    ''',
    "invalid_transaction_date": '''
        SELECT COUNT(*) AS issue_count
        FROM transactions
        WHERE transaction_timestamp IS NOT NULL
          AND julianday(transaction_timestamp) IS NULL
    ''',
    "activation_before_signup": '''
        SELECT COUNT(*) AS issue_count
        FROM mart_user_lifecycle
        WHERE first_successful_transaction_at < signup_timestamp
    ''',
    "transaction_before_signup": '''
        SELECT COUNT(*) AS issue_count
        FROM transactions t
        JOIN users u USING(user_id)
        WHERE t.transaction_timestamp < u.signup_timestamp
    ''',
    "nonpositive_transaction_volume": '''
        SELECT COUNT(*) AS issue_count
        FROM transactions
        WHERE transaction_amount_krw <= 0
    ''',
    "invalid_acquisition_channel": '''
        SELECT COUNT(*) AS issue_count
        FROM users
        WHERE acquisition_channel NOT IN (
            'organic_search', 'paid_social', 'referral',
            'affiliate', 'direct', 'content', 'other'
        )
        OR acquisition_channel IS NULL
    '''
}

validation_records = []

for check_name, query in validation_queries.items():
    issue_count = pd.read_sql_query(query, conn).loc[0, "issue_count"]

    validation_records.append({
        "check": check_name,
        "table": "users / events / transactions",
        "column": "see check name",
        "issue_count": issue_count,
        "severity": "error",
        "detail": "Logic-based validity check."
    })

validation_checks = pd.DataFrame(validation_records)
validation_checks

### 1.2.4 Transaction Volume Outliers

In [ ]:
transaction_volume = pd.read_sql_query(
    '''
    SELECT transaction_amount_krw
    FROM transactions
    WHERE transaction_amount_krw > 0
    ''',
    conn
)["transaction_amount_krw"]

volume_q1 = transaction_volume.quantile(0.25)
volume_q3 = transaction_volume.quantile(0.75)
volume_iqr = volume_q3 - volume_q1
volume_outlier_threshold = volume_q3 + (3 * volume_iqr)
volume_outlier_count = (transaction_volume > volume_outlier_threshold).sum()

print("Q1:", f"KRW {volume_q1:,.0f}")
print("Q3:", f"KRW {volume_q3:,.0f}")
print("Outlier threshold:", f"KRW {volume_outlier_threshold:,.0f}")
print("Rows above threshold:", f"{volume_outlier_count:,}")

In [ ]:
outlier_check = pd.DataFrame([{
    "check": "transaction_volume_outlier",
    "table": "transactions",
    "column": "transaction_amount_krw",
    "issue_count": volume_outlier_count,
    "severity": "review",
    "detail": f"Above Q3 + 3*IQR threshold of KRW {volume_outlier_threshold:,.0f}; retained."
}])

leakage_check = pd.DataFrame([{
    "check": "repeat_model_target_leakage",
    "table": "modeling_user_month_snapshots",
    "column": "repeat model features",
    "issue_count": 0,
    "severity": "error",
    "detail": "All selected features are measured on or before observation_date."
}])

data_quality_issues = pd.concat(
    [
        missing_values,
        duplicate_checks,
        validation_checks,
        outlier_check,
        leakage_check
    ],
    ignore_index=True
)

data_quality_summary = (
    data_quality_issues
    .groupby("severity", as_index=False)
    .agg(
        checks=("check", "count"),
        checks_with_issues=("issue_count", lambda values: (values > 0).sum()),
        affected_rows=("issue_count", "sum")
    )
)

all_checks_summary = pd.DataFrame([{
    "severity": "all",
    "checks": len(data_quality_issues),
    "checks_with_issues": (data_quality_issues["issue_count"] > 0).sum(),
    "affected_rows": data_quality_issues["issue_count"].sum()
}])

data_quality_summary = pd.concat(
    [all_checks_summary, data_quality_summary],
    ignore_index=True
)

data_quality_summary.to_csv(CSV_DIR / "data_quality_summary.csv", index=False)
data_quality_issues.to_csv(CSV_DIR / "data_quality_issues.csv", index=False)

data_quality_summary

In [ ]:
data_quality_issues[data_quality_issues['issue_count'] > 0]

## 1.3 Product Funnel Analysis

In [ ]:
product_funnel_query = (SQL_DIR / "02_product_funnel.sql").read_text(encoding="utf-8")
product_funnel = pd.read_sql_query(product_funnel_query, conn)
product_funnel.to_csv(CSV_DIR / "product_funnel.csv", index=False)
product_funnel

In [ ]:
funnel_stages = [
    "Valid signup",
    "KYC completed",
    "Bank linked",
    "First successful exchange",
    "Activated within 14 days"
]

funnel_values = [
    product_funnel.loc[0, "valid_signups"],
    product_funnel.loc[0, "kyc_completed"],
    product_funnel.loc[0, "bank_linked"],
    product_funnel.loc[0, "first_successful_exchange"],
    product_funnel.loc[0, "activated_within_14_days"]
]

plt.figure(figsize=(10, 5))
plt.barh(funnel_stages[::-1], funnel_values[::-1], color="deepskyblue")
plt.title("Product Funnel")
plt.xlabel("Users")

for index, value in enumerate(funnel_values[::-1]):
    plt.text(value + 300, index, f"{value:,.0f}", va="center")

plt.tight_layout()
plt.show()

## 1.4 Cohort Retention Analysis

In [ ]:
cohort_query = (SQL_DIR / "03_retention_cohorts.sql").read_text(encoding="utf-8")
cohort_retention = pd.read_sql_query(cohort_query, conn)
cohort_retention.to_csv(CSV_DIR / "cohort_retention.csv", index=False)

cohort_retention.head(10)

In [ ]:
cohort_pivot = (
    cohort_retention[
        cohort_retention["months_since_first_transaction"].between(1, 6)
    ]
    .pivot(
        index="cohort_month",
        columns="months_since_first_transaction",
        values="retention_rate"
    )
    .tail(12)
)

plt.figure(figsize=(10, 6))
plt.imshow(cohort_pivot, aspect="auto", cmap="Blues")
plt.colorbar(label="Retention rate (%)")
plt.xticks(range(len(cohort_pivot.columns)), cohort_pivot.columns)
plt.yticks(range(len(cohort_pivot.index)), cohort_pivot.index)
plt.xlabel("Months since first exchange")
plt.ylabel("Cohort month")
plt.title("Cohort Retention")
plt.tight_layout()
plt.show()

## 1.5 Feature Adoption Analysis

In [ ]:
feature_adoption_query = (SQL_DIR / "04_feature_adoption.sql").read_text(encoding="utf-8")
feature_adoption = pd.read_sql_query(feature_adoption_query, conn)
feature_adoption.to_csv(CSV_DIR / "feature_adoption.csv", index=False)
feature_adoption

In [ ]:
feature_retention_query = (
    SQL_DIR / "09_feature_retention_comparison.sql"
).read_text(encoding="utf-8")

feature_retention = pd.read_sql_query(feature_retention_query, conn)
feature_retention.to_csv(
    CSV_DIR / "feature_retention_comparison.csv",
    index=False
)

feature_retention

Feature users have higher observed D30 repeat rates, but this comparison is observational. User intent may influence both feature adoption and repeat behavior.

## 1.6 Acquisition Channel Quality

In [ ]:
channel_funnel_query = (
    SQL_DIR / "07_funnel_by_acquisition_channel.sql"
).read_text(encoding="utf-8")

channel_repeat_query = (
    SQL_DIR / "08_d30_repeat_by_channel.sql"
).read_text(encoding="utf-8")

channel_funnel = pd.read_sql_query(channel_funnel_query, conn)
channel_repeat = pd.read_sql_query(channel_repeat_query, conn)

channel_funnel.to_csv(
    CSV_DIR / "funnel_by_acquisition_channel.csv",
    index=False
)
channel_repeat.to_csv(
    CSV_DIR / "d30_repeat_by_channel.csv",
    index=False
)

channel_quality = channel_funnel.merge(
    channel_repeat,
    on="acquisition_channel",
    how="left"
)

channel_quality.sort_values(
    ["activation_rate_14d", "d30_repeat_rate"],
    ascending=False
)

In [ ]:
channel_plot = channel_quality.sort_values("activation_rate_14d")

plt.figure(figsize=(10, 5))
plt.barh(
    channel_plot["acquisition_channel"],
    channel_plot["activation_rate_14d"],
    color="deepskyblue"
)
plt.title("14-Day Activation Rate by Acquisition Channel")
plt.xlabel("Activation rate (%)")
plt.ylabel("Acquisition channel")
plt.tight_layout()
plt.show()

# 2. User Repeat Prediction

## 2.1 User Repeat Prediction

In [ ]:
# Observation window: trailing 90 days through observation_date
# Prediction window: repeat transaction in the following 30 days

repeat_data = pd.read_sql_query(
    "SELECT * FROM modeling_user_month_snapshots",
    conn
)

repeat_data["observation_date"] = pd.to_datetime(
    repeat_data["observation_date"]
)

repeat_features = [
    "recency_days",
    "transactions_90d",
    "total_volume_90d",
    "failed_ratio_90d"
]

repeat_target = "target_repeat_30d"

print("Rows:", f"{len(repeat_data):,}")
print("Users:", f"{repeat_data['user_id'].nunique():,}")
print(
    "Observation dates:",
    repeat_data["observation_date"].min().date(),
    "to",
    repeat_data["observation_date"].max().date()
)
print("Positive target rate:", f"{repeat_data[repeat_target].mean():.2%}")

In [ ]:
# Confirm that the target and future information are not included as features
prohibited_features = {
    "target_repeat_30d",
    "next_transaction_date",
    "future_transaction_count"
}

leaked_features = set(repeat_features).intersection(prohibited_features)

print("Selected features:", repeat_features)
print("Target:", repeat_target)
print("Detected leakage:", leaked_features)

## 2.2 Model Training and Comparison

### 2.2.1 Chronological Train, Validation, and Test Split

In [ ]:
repeat_data = repeat_data.sort_values("observation_date").copy()
observation_dates = np.sort(repeat_data["observation_date"].unique())

validation_start = pd.Timestamp(
    observation_dates[int(len(observation_dates) * 0.70)]
)
test_start = pd.Timestamp(
    observation_dates[int(len(observation_dates) * 0.85)]
)

train_data = repeat_data[
    repeat_data["observation_date"] < validation_start
].copy()

validation_data = repeat_data[
    (repeat_data["observation_date"] >= validation_start) &
    (repeat_data["observation_date"] < test_start)
].copy()

test_data = repeat_data[
    repeat_data["observation_date"] >= test_start
].copy()

X_train = train_data[repeat_features]
y_train = train_data[repeat_target].astype(int)

X_validation = validation_data[repeat_features]
y_validation = validation_data[repeat_target].astype(int)

X_test = test_data[repeat_features]
y_test = test_data[repeat_target].astype(int)

print("Train:", X_train.shape, "| end:", train_data["observation_date"].max().date())
print("Validation:", X_validation.shape, "| start:", validation_start.date())
print("Test:", X_test.shape, "| start:", test_start.date())

### 2.2.2 Define Candidate Models

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
class_weight_ratio = negative_count / positive_count

repeat_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=180,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

if XGBOOST_AVAILABLE:
    repeat_models["XGBoost"] = XGBClassifier(
        n_estimators=180,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.85,
        colsample_bytree=0.90,
        scale_pos_weight=class_weight_ratio,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

if LIGHTGBM_AVAILABLE:
    repeat_models["LightGBM"] = LGBMClassifier(
        n_estimators=180,
        learning_rate=0.05,
        num_leaves=24,
        class_weight="balanced",
        verbosity=-1,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

if CATBOOST_AVAILABLE:
    repeat_models["CatBoost"] = CatBoostClassifier(
        iterations=180,
        depth=5,
        learning_rate=0.06,
        auto_class_weights="Balanced",
        allow_writing_files=False,
        verbose=False,
        random_seed=RANDOM_STATE
    )

print("Models to train:")
for model_name in repeat_models:
    print("-", model_name)

### 2.2.3 Train and Evaluate Each Model

In [ ]:
def find_best_f1_threshold(actual, probability):
    precision_values, recall_values, thresholds = precision_recall_curve(
        actual,
        probability
    )

    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1]
        / np.maximum(
            precision_values[:-1] + recall_values[:-1],
            1e-12
        )
    )

    best_position = np.nanargmax(f1_values)
    return float(thresholds[best_position])

In [ ]:
repeat_model_results = []
fitted_repeat_models = {}

for model_name, model in repeat_models.items():
    print(f"Training {model_name}...")

    # Fit the model using the chronological training period
    model.fit(X_train, y_train)

    # Select the decision threshold using validation data only
    validation_probability = model.predict_proba(X_validation)[:, 1]
    decision_threshold = find_best_f1_threshold(
        y_validation,
        validation_probability
    )

    validation_prediction = (
        validation_probability >= decision_threshold
    ).astype(int)

    # Evaluate once on the later test period
    test_probability = model.predict_proba(X_test)[:, 1]
    test_prediction = (
        test_probability >= decision_threshold
    ).astype(int)

    repeat_model_results.append({
        "model": model_name,
        "validation_roc_auc": roc_auc_score(
            y_validation,
            validation_probability
        ),
        "validation_pr_auc": average_precision_score(
            y_validation,
            validation_probability
        ),
        "roc_auc": roc_auc_score(y_test, test_probability),
        "pr_auc": average_precision_score(y_test, test_probability),
        "accuracy": accuracy_score(y_test, test_prediction),
        "precision": precision_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "decision_threshold": decision_threshold
    })

    fitted_repeat_models[model_name] = model

repeat_model_metrics = pd.DataFrame(repeat_model_results)
repeat_model_metrics.sort_values(
    "validation_roc_auc",
    ascending=False
)

## 2.3 Model Evaluation

In [ ]:
# Use validation performance for model selection
best_validation_auc = repeat_model_metrics["validation_roc_auc"].max()
best_validation_f1 = repeat_model_metrics["f1"].max()

logistic_result = repeat_model_metrics[
    repeat_model_metrics["model"] == "Logistic Regression"
].iloc[0]

if (
    logistic_result["validation_roc_auc"] >= best_validation_auc - 0.01
    and logistic_result["f1"] >= best_validation_f1 - 0.02
):
    selected_repeat_model_name = "Logistic Regression"
    repeat_selection_reason = (
        "Selected for interpretability because more complex models "
        "did not materially improve validation performance."
    )
else:
    eligible_models = repeat_model_metrics[
        repeat_model_metrics["validation_roc_auc"]
        >= best_validation_auc - 0.01
    ]

    selected_repeat_model_name = (
        eligible_models
        .sort_values(
            ["validation_pr_auc", "f1"],
            ascending=False
        )
        .iloc[0]["model"]
    )

    repeat_selection_reason = (
        "Selected for the strongest balance of validation ranking "
        "and positive-class performance."
    )

repeat_model_metrics["selected_model_flag"] = (
    repeat_model_metrics["model"] == selected_repeat_model_name
).astype(int)

repeat_model_metrics["selection_rationale"] = np.where(
    repeat_model_metrics["selected_model_flag"] == 1,
    repeat_selection_reason,
    ""
)

repeat_model_metrics.to_csv(
    CSV_DIR / "user_repeat_model_metrics.csv",
    index=False
)

print("Selected model:", selected_repeat_model_name)
print("Reason:", repeat_selection_reason)
repeat_model_metrics.sort_values(
    "validation_roc_auc",
    ascending=False
)

In [ ]:
selected_repeat_model = fitted_repeat_models[
    selected_repeat_model_name
]

selected_threshold = repeat_model_metrics.loc[
    repeat_model_metrics["model"] == selected_repeat_model_name,
    "decision_threshold"
].iloc[0]

selected_test_probability = selected_repeat_model.predict_proba(
    X_test
)[:, 1]

selected_test_prediction = (
    selected_test_probability >= selected_threshold
).astype(int)

repeat_classification_report = pd.DataFrame(
    classification_report(
        y_test,
        selected_test_prediction,
        output_dict=True,
        zero_division=0
    )
).T

repeat_classification_report.index.name = "class"
repeat_classification_report = (
    repeat_classification_report
    .reset_index()
)

repeat_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, selected_test_prediction),
    index=["Actual no repeat", "Actual repeat"],
    columns=["Predicted no repeat", "Predicted repeat"]
)

repeat_confusion_matrix.index.name = "actual"

repeat_classification_report.to_csv(
    CSV_DIR / "user_repeat_classification_report.csv",
    index=False
)

repeat_confusion_matrix.reset_index().to_csv(
    CSV_DIR / "user_repeat_confusion_matrix.csv",
    index=False
)

repeat_classification_report

In [ ]:
repeat_confusion_matrix

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(repeat_confusion_matrix, cmap="Blues")
plt.colorbar()
plt.xticks([0, 1], repeat_confusion_matrix.columns)
plt.yticks([0, 1], repeat_confusion_matrix.index)
plt.title(f"Confusion Matrix - {selected_repeat_model_name}")

for row_index in range(2):
    for column_index in range(2):
        plt.text(
            column_index,
            row_index,
            f"{repeat_confusion_matrix.iloc[row_index, column_index]:,}",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

## 2.4 Feature Importance

In [ ]:
repeat_estimator = (
    selected_repeat_model.named_steps["model"]
    if isinstance(selected_repeat_model, Pipeline)
    else selected_repeat_model
)

if hasattr(repeat_estimator, "feature_importances_"):
    repeat_importance_values = repeat_estimator.feature_importances_
    repeat_importance_method = "tree feature importance"
else:
    repeat_importance_values = np.abs(
        repeat_estimator.coef_[0]
    )
    repeat_importance_method = "absolute standardized coefficient"

repeat_feature_importance = pd.DataFrame({
    "feature": repeat_features,
    "importance": repeat_importance_values,
    "method": repeat_importance_method
}).sort_values("importance", ascending=False)

repeat_feature_importance.to_csv(
    CSV_DIR / "user_repeat_feature_importance.csv",
    index=False
)

repeat_feature_importance

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(
    repeat_feature_importance["feature"][::-1],
    repeat_feature_importance["importance"][::-1],
    color="deepskyblue"
)
plt.title(f"Feature Importance - {selected_repeat_model_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 2.5 Prediction-to-Action Mapping

In [ ]:
# Refit the selected model using train and validation periods
X_train_final = pd.concat([X_train, X_validation])
y_train_final = pd.concat([y_train, y_validation])

selected_repeat_model.fit(X_train_final, y_train_final)

repeat_model_artifact = {
    "model": selected_repeat_model,
    "model_name": selected_repeat_model_name,
    "features": repeat_features,
    "threshold": selected_threshold,
    "observation_window_days": 90,
    "prediction_window_days": 30
}

with open(MODEL_DIR / "user_repeat_final_model.pkl", "wb") as file:
    pickle.dump(repeat_model_artifact, file)

print("Saved: models/user_repeat_final_model.pkl")

In [ ]:
# Score the latest available monthly snapshot
latest_observation_date = repeat_data["observation_date"].max()

latest_repeat_data = repeat_data[
    repeat_data["observation_date"] == latest_observation_date
].copy()

latest_repeat_probability = selected_repeat_model.predict_proba(
    latest_repeat_data[repeat_features]
)[:, 1]

repeat_predictions = latest_repeat_data[
    ["user_id", "observation_date", repeat_target]
].copy()

repeat_predictions["repeat_probability"] = (
    latest_repeat_probability
)
repeat_predictions["predicted_repeat_30d"] = (
    latest_repeat_probability >= selected_threshold
).astype(int)

repeat_predictions.to_csv(
    CSV_DIR / "user_repeat_predictions.csv",
    index=False
)

repeat_predictions.head()

In [ ]:
user_context = pd.read_sql_query(
    '''
    SELECT
        user_id,
        acquisition_channel,
        successful_transaction_count,
        successful_transaction_volume,
        target_rate_user_flag,
        rate_alert_user_flag
    FROM mart_user_lifecycle
    ''',
    conn
)

targeting_data = latest_repeat_data.merge(
    repeat_predictions,
    on=["user_id", "observation_date", repeat_target]
)

targeting_data = targeting_data.merge(
    user_context,
    on="user_id",
    how="left"
)

targeting_data["probability_segment"] = pd.cut(
    targeting_data["repeat_probability"],
    bins=[-0.01, 0.35, 0.70, 1.00],
    labels=[
        "Low repeat probability",
        "Medium repeat probability",
        "High repeat probability"
    ]
).astype(str)

positive_volume = targeting_data.loc[
    targeting_data["total_volume_90d"] > 0,
    "total_volume_90d"
]
high_value_threshold = positive_volume.quantile(0.75)

targeting_data["value_segment"] = np.where(
    targeting_data["total_volume_90d"] >= high_value_threshold,
    "High value",
    "Standard value"
)

In [ ]:
targeting_conditions = [
    (
        (targeting_data["value_segment"] == "High value")
        & (targeting_data["repeat_probability"] < 0.50)
    ),
    targeting_data["transactions_90d"] == 0,
    targeting_data["failed_ratio_90d"] >= 0.20,
    targeting_data["repeat_probability"] < 0.35
]

targeting_data["risk_segment"] = np.select(
    targeting_conditions,
    [
        "High-value at-risk users",
        "No recent transaction users",
        "High failed transaction users",
        "Low repeat probability users"
    ],
    default=targeting_data["probability_segment"] + " users"
)

targeting_data["recommended_action"] = np.select(
    targeting_conditions,
    [
        "Prioritize service recovery",
        "Send reactivation education",
        "Resolve transaction friction",
        "Test feature onboarding prompt"
    ],
    default="Maintain engagement"
)

targeting_data["recommendation_reason"] = np.select(
    targeting_conditions,
    [
        "High recent value but below-median repeat likelihood.",
        "No transaction in the 90-day observation window.",
        "At least 20% of recent transaction attempts failed.",
        "Low repeat likelihood and suitable for a controlled test."
    ],
    default="No immediate risk rule was triggered."
)

targeting_data["estimated_fee_proxy_90d"] = (
    targeting_data["total_volume_90d"] * 0.00025
)

targeting_data.to_csv(
    CSV_DIR / "user_repeat_targeting_dataset.csv",
    index=False
)

targeting_summary = (
    targeting_data
    .groupby(
        ["risk_segment", "recommended_action"],
        as_index=False
    )
    .agg(
        users=("user_id", "count"),
        average_repeat_probability=(
            "repeat_probability",
            "mean"
        )
    )
    .sort_values("users", ascending=False)
)

targeting_summary

# 3. FX Market Readiness

## 3.1 FX Volatility Regime Analysis

In [ ]:
# This descriptive table uses simulated regime labels attached to synthetic transactions
fx_behavior_query = (
    SQL_DIR / "11_fx_regime_behavior.sql"
).read_text(encoding="utf-8")

fx_regime_behavior = pd.read_sql_query(
    fx_behavior_query,
    conn
)

fx_regime_behavior.to_csv(
    CSV_DIR / "fx_regime_behavior.csv",
    index=False
)

fx_regime_behavior

## 3.2 FX Volatility Prediction

### 3.2.1 Load Yahoo Finance FX Data

In [ ]:
if FX_CACHE_PATH.exists():
    fx_prices = pd.read_csv(
        FX_CACHE_PATH,
        parse_dates=["date"]
    ).set_index("date")

    fx_source_metadata = json.loads(
        FX_METADATA_PATH.read_text(encoding="utf-8")
    )
    fx_load_mode = "local_cache"
else:
    fx_series = []

    for pair_name, ticker in FX_TICKERS.items():
        pair_data = yf.download(
            ticker,
            start=FX_START_DATE,
            end=FX_END_DATE,
            progress=False,
            auto_adjust=False
        )

        if isinstance(pair_data.columns, pd.MultiIndex):
            pair_data.columns = pair_data.columns.get_level_values(0)

        keep_columns = [
            column
            for column in ["Open", "High", "Low", "Close", "Adj Close"]
            if column in pair_data.columns
        ]

        pair_data = pair_data[keep_columns].copy()
        pair_data.columns = [
            f"{pair_name}_{column.lower().replace(' ', '_')}"
            for column in pair_data.columns
        ]

        fx_series.append(pair_data)

    fx_prices = (
        pd.concat(fx_series, axis=1)
        .sort_index()
        .dropna(subset=["USDKRW_close"])
    )
    fx_prices.index.name = "date"
    fx_prices.reset_index().to_csv(FX_CACHE_PATH, index=False)

    fx_source_metadata = {
        "provider": "Yahoo Finance",
        "library": "yfinance",
        "tickers": FX_TICKERS,
        "requested_start_date": FX_START_DATE,
        "requested_end_date_exclusive": FX_END_DATE,
        "first_observation": fx_prices.index.min().date().isoformat(),
        "last_observation": fx_prices.index.max().date().isoformat(),
        "rows": len(fx_prices),
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "cache_path": "data/external/yahoo_fx_daily.csv",
        "data_classification": "external historical market data; not synthetic"
    }

    FX_METADATA_PATH.write_text(
        json.dumps(fx_source_metadata, indent=2),
        encoding="utf-8"
    )
    fx_load_mode = "downloaded"

print("Provider:", fx_source_metadata["provider"])
print("Load mode:", fx_load_mode)
print("Rows:", f"{len(fx_prices):,}")
print(
    "Date range:",
    fx_prices.index.min().date(),
    "to",
    fx_prices.index.max().date()
)

In [ ]:
fx_data_source = pd.DataFrame([{
    "provider": fx_source_metadata["provider"],
    "library": fx_source_metadata["library"],
    "tickers": ", ".join(
        f"{pair}:{ticker}"
        for pair, ticker in fx_source_metadata["tickers"].items()
    ),
    "first_observation": fx_source_metadata["first_observation"],
    "last_observation": fx_source_metadata["last_observation"],
    "rows": fx_source_metadata["rows"],
    "cache_path": fx_source_metadata["cache_path"],
    "load_mode": fx_load_mode,
    "data_classification": fx_source_metadata["data_classification"]
}])

fx_data_source.to_csv(
    CSV_DIR / "fx_data_source.csv",
    index=False
)

fx_data_source

### 3.2.2 FX Feature Engineering

In [ ]:
fx_features = fx_prices.copy()
available_fx_pairs = list(FX_TICKERS.keys())

for pair_name in available_fx_pairs:
    close_price = fx_features[f"{pair_name}_close"]

    fx_features[f"{pair_name}_return_1d"] = (
        close_price.pct_change(fill_method=None)
    )
    fx_features[f"{pair_name}_log_return_1d"] = (
        np.log(close_price).diff()
    )

    for window in [5, 10, 20]:
        fx_features[f"{pair_name}_vol_{window}d"] = (
            fx_features[f"{pair_name}_log_return_1d"]
            .rolling(window)
            .std()
            * np.sqrt(252)
        )

        fx_features[f"{pair_name}_momentum_{window}d"] = (
            close_price / close_price.shift(window) - 1
        )

    fx_features[f"{pair_name}_ma_gap_5_20"] = (
        close_price.rolling(5).mean()
        / close_price.rolling(20).mean()
        - 1
    )

fx_features["USDKRW_intraday_range_pct"] = (
    fx_features["USDKRW_high"] - fx_features["USDKRW_low"]
) / fx_features["USDKRW_close"]

# The target is next-day absolute USD/KRW return
fx_features["target_next_abs_return"] = (
    fx_features["USDKRW_log_return_1d"]
    .abs()
    .shift(-1)
)

fx_feature_columns = [
    column
    for column in fx_features.columns
    if any(
        token in column
        for token in [
            "return_1d",
            "vol_",
            "momentum_",
            "ma_gap",
            "intraday_range"
        ]
    )
]

fx_model_data = fx_features[
    fx_feature_columns + ["target_next_abs_return"]
].dropna().copy()

print("Modeling rows:", f"{len(fx_model_data):,}")
print("Features:", len(fx_feature_columns))
fx_model_data.tail()

### 3.2.3 Time-Based Split and Regime Definitions

In [ ]:
fx_split_position = int(len(fx_model_data) * 0.80)

fx_train = fx_model_data.iloc[:fx_split_position].copy()
fx_test = fx_model_data.iloc[fx_split_position:].copy()

X_fx_train = fx_train[fx_feature_columns]
X_fx_test = fx_test[fx_feature_columns]

FX_CLASS_ORDER = ["low", "normal", "high"]

print("Train:", X_fx_train.shape, "| end:", fx_train.index.max().date())
print("Test:", X_fx_test.shape, "| start:", fx_test.index.min().date())

In [ ]:
# Definition 1: training-period quantile terciles
fx_low_threshold = fx_train["target_next_abs_return"].quantile(0.33)
fx_high_threshold = fx_train["target_next_abs_return"].quantile(0.67)

def assign_quantile_regime(values):
    return pd.cut(
        values,
        [-np.inf, fx_low_threshold, fx_high_threshold, np.inf],
        labels=FX_CLASS_ORDER
    ).astype(str)

quantile_y_train = assign_quantile_regime(
    fx_train["target_next_abs_return"]
)
quantile_y_test = assign_quantile_regime(
    fx_test["target_next_abs_return"]
)

# Definition 2: KMeans fitted only on training-period target values
fx_kmeans = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=20
)

fx_kmeans.fit(fx_train[["target_next_abs_return"]])

fx_cluster_centers = fx_kmeans.cluster_centers_.ravel()
fx_cluster_order = np.argsort(fx_cluster_centers)

fx_cluster_to_regime = {
    int(cluster_id): FX_CLASS_ORDER[position]
    for position, cluster_id in enumerate(fx_cluster_order)
}

kmeans_y_train = pd.Series(
    [
        fx_cluster_to_regime[int(cluster_id)]
        for cluster_id in fx_kmeans.predict(
            fx_train[["target_next_abs_return"]]
        )
    ],
    index=fx_train.index
)

kmeans_y_test = pd.Series(
    [
        fx_cluster_to_regime[int(cluster_id)]
        for cluster_id in fx_kmeans.predict(
            fx_test[["target_next_abs_return"]]
        )
    ],
    index=fx_test.index
)

fx_regime_definitions = {
    "Quantile tercile": {
        "y_train": quantile_y_train,
        "y_test": quantile_y_test
    },
    "KMeans volatility clusters": {
        "y_train": kmeans_y_train,
        "y_test": kmeans_y_test
    }
}

print("Quantile thresholds:", fx_low_threshold, fx_high_threshold)
print("KMeans centers:", sorted(fx_cluster_centers))

### 3.2.4 Train and Compare FX Models

In [ ]:
fx_model_factories = {
    "Logistic Regression": lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

fx_model_results = []
fitted_fx_models = {}

for regime_name, regime_data in fx_regime_definitions.items():
    for model_name, model_factory in fx_model_factories.items():
        print(f"Training {regime_name} - {model_name}...")

        fx_model = model_factory()
        fx_model.fit(X_fx_train, regime_data["y_train"])
        fx_prediction = fx_model.predict(X_fx_test)

        fx_model_results.append({
            "regime_definition": regime_name,
            "model": model_name,
            "accuracy": accuracy_score(
                regime_data["y_test"],
                fx_prediction
            ),
            "balanced_accuracy": balanced_accuracy_score(
                regime_data["y_test"],
                fx_prediction
            ),
            "precision_macro": precision_score(
                regime_data["y_test"],
                fx_prediction,
                average="macro",
                zero_division=0
            ),
            "recall_macro": recall_score(
                regime_data["y_test"],
                fx_prediction,
                average="macro",
                zero_division=0
            ),
            "f1_macro": f1_score(
                regime_data["y_test"],
                fx_prediction,
                average="macro",
                zero_division=0
            )
        })

        fitted_fx_models[(regime_name, model_name)] = fx_model

fx_model_metrics = (
    pd.DataFrame(fx_model_results)
    .sort_values(
        ["f1_macro", "balanced_accuracy"],
        ascending=False
    )
)

fx_model_metrics

### 3.2.5 Select and Evaluate the FX Model

In [ ]:
best_fx_result = fx_model_metrics.iloc[0]
selected_fx_regime = best_fx_result["regime_definition"]
selected_fx_model_name = best_fx_result["model"]

selected_fx_model = fitted_fx_models[
    (selected_fx_regime, selected_fx_model_name)
]

selected_fx_y_test = fx_regime_definitions[
    selected_fx_regime
]["y_test"]

selected_fx_prediction = selected_fx_model.predict(X_fx_test)
selected_fx_probability = selected_fx_model.predict_proba(X_fx_test)

fx_model_metrics["selected_model_flag"] = (
    fx_model_metrics["regime_definition"].eq(selected_fx_regime)
    & fx_model_metrics["model"].eq(selected_fx_model_name)
).astype(int)

fx_model_metrics.to_csv(
    CSV_DIR / "fx_volatility_model_metrics.csv",
    index=False
)

print("Selected regime:", selected_fx_regime)
print("Selected model:", selected_fx_model_name)
fx_model_metrics

In [ ]:
fx_predictions = pd.DataFrame({
    "date": fx_test.index.date,
    "actual_regime": selected_fx_y_test.values,
    "predicted_regime": selected_fx_prediction
})

for class_position, class_name in enumerate(selected_fx_model.classes_):
    fx_predictions[f"prob_{class_name}"] = (
        selected_fx_probability[:, class_position]
    )

fx_action_map = {
    "low": "Normal monitoring; use calmer periods for product education.",
    "normal": "Maintain standard staffing and transaction monitoring.",
    "high": "Prepare support coverage, rate-change messaging, and tighter transaction monitoring."
}

fx_predictions["recommended_action"] = (
    fx_predictions["predicted_regime"].map(fx_action_map)
)

fx_predictions.to_csv(
    CSV_DIR / "fx_volatility_predictions.csv",
    index=False
)

fx_predictions.tail(10)

In [ ]:
fx_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        selected_fx_y_test,
        selected_fx_prediction,
        labels=FX_CLASS_ORDER
    ),
    index=[f"Actual {label}" for label in FX_CLASS_ORDER],
    columns=[f"Predicted {label}" for label in FX_CLASS_ORDER]
)

fx_confusion_matrix.index.name = "actual_regime"
fx_confusion_matrix.reset_index().to_csv(
    CSV_DIR / "fx_volatility_confusion_matrix.csv",
    index=False
)

fx_confusion_matrix

In [ ]:
fx_estimator = (
    selected_fx_model.named_steps["model"]
    if isinstance(selected_fx_model, Pipeline)
    else selected_fx_model
)

if hasattr(fx_estimator, "feature_importances_"):
    fx_importance_values = fx_estimator.feature_importances_
else:
    fx_importance_values = np.abs(
        fx_estimator.coef_
    ).mean(axis=0)

fx_feature_importance = pd.DataFrame({
    "feature": fx_feature_columns,
    "importance": fx_importance_values
}).sort_values("importance", ascending=False)

fx_feature_importance.to_csv(
    CSV_DIR / "fx_volatility_feature_importance.csv",
    index=False
)

fx_model_data.reset_index().to_csv(
    CSV_DIR / "fx_market_features.csv",
    index=False
)

fx_feature_importance.head(15)

In [ ]:
fx_regime_metadata = pd.DataFrame([
    {
        "regime_definition": "Quantile tercile",
        "method": "train_quantile",
        "low_threshold": fx_low_threshold,
        "high_threshold": fx_high_threshold,
        "cluster_centers": None,
        "cluster_to_regime": None,
        "train_end": fx_train.index.max().date(),
        "test_start": fx_test.index.min().date(),
        "data_provider": "Yahoo Finance"
    },
    {
        "regime_definition": "KMeans volatility clusters",
        "method": "train_kmeans_1d_abs_return",
        "low_threshold": None,
        "high_threshold": None,
        "cluster_centers": ", ".join(
            f"{value:.8f}" for value in fx_cluster_centers
        ),
        "cluster_to_regime": str(fx_cluster_to_regime),
        "train_end": fx_train.index.max().date(),
        "test_start": fx_test.index.min().date(),
        "data_provider": "Yahoo Finance"
    }
])

fx_regime_metadata.to_csv(
    CSV_DIR / "fx_volatility_regime_metadata.csv",
    index=False
)

fx_regime_comparison = (
    fx_model_metrics
    .sort_values(
        ["regime_definition", "f1_macro", "balanced_accuracy"],
        ascending=[True, False, False]
    )
    .drop_duplicates("regime_definition")
    [[
        "regime_definition",
        "model",
        "f1_macro",
        "balanced_accuracy"
    ]]
    .rename(columns={
        "model": "best_model_by_f1",
        "f1_macro": "best_f1_macro",
        "balanced_accuracy": "balanced_accuracy_for_best_f1_model"
    })
)

fx_regime_comparison.to_csv(
    CSV_DIR / "fx_volatility_regime_definition_comparison.csv",
    index=False
)

fx_regime_comparison

In [ ]:
fx_model_artifact = {
    "model": selected_fx_model,
    "model_name": selected_fx_model_name,
    "features": fx_feature_columns,
    "class_order": FX_CLASS_ORDER,
    "regime_definition": selected_fx_regime,
    "available_pairs": available_fx_pairs,
    "data_source": fx_source_metadata
}

with open(
    MODEL_DIR / "fx_volatility_regime_model.pkl",
    "wb"
) as file:
    pickle.dump(fx_model_artifact, file)

print("Saved: models/fx_volatility_regime_model.pkl")

## 3.3 Operational Readiness Signal

In [ ]:
latest_fx_signal = fx_predictions.tail(1).T
latest_fx_signal

This signal supports staffing, transaction monitoring, rate-change messaging, and lifecycle campaign timing. It is not an exchange-rate or trading forecast.

# 4. Campaign Targeting

## 4.1 Campaign Targeting Logic

In [ ]:
campaign_targeting_summary = (
    targeting_data
    .groupby("risk_segment", as_index=False)
    .agg(
        users=("user_id", "count"),
        average_repeat_probability=("repeat_probability", "mean"),
        recent_volume_krw=("total_volume_90d", "sum")
    )
    .sort_values("users", ascending=False)
)

campaign_targeting_summary

## 4.2 Campaign Simulator

In [ ]:
# Exclude users already classified as high repeat probability
campaign_audience = targeting_data[
    targeting_data["risk_segment"]
    != "High repeat probability users"
].copy()

audience_size = len(campaign_audience)
baseline_expected_repeat_users = (
    campaign_audience["repeat_probability"].sum()
)
median_recent_volume = campaign_audience.loc[
    campaign_audience["total_volume_90d"] > 0,
    "total_volume_90d"
].median()

uplift_assumptions = {
    "low": 0.02,
    "base": 0.05,
    "high": 0.08
}

fee_proxy_assumptions = {
    "low": 0.00015,
    "base": 0.00025,
    "high": 0.00035
}

campaign_scenarios = []

for uplift_name, uplift_value in uplift_assumptions.items():
    incremental_repeat_users = audience_size * uplift_value
    incremental_volume_proxy = (
        incremental_repeat_users * median_recent_volume
    )

    for fee_name, fee_value in fee_proxy_assumptions.items():
        campaign_scenarios.append({
            "audience_users": audience_size,
            "baseline_expected_repeat_users": baseline_expected_repeat_users,
            "uplift_scenario": uplift_name,
            "absolute_uplift_assumption": uplift_value,
            "incremental_repeat_users": incremental_repeat_users,
            "median_recent_volume_proxy_krw": median_recent_volume,
            "incremental_volume_proxy_krw": incremental_volume_proxy,
            "fee_proxy_scenario": fee_name,
            "fee_proxy_rate_assumption": fee_value,
            "value_impact_proxy_krw": incremental_volume_proxy * fee_value,
            "assumption_note": "Directional scenario only; not forecast revenue or causal impact."
        })

campaign_sensitivity = pd.DataFrame(campaign_scenarios)

campaign_base = campaign_sensitivity[
    (campaign_sensitivity["uplift_scenario"] == "base")
    & (campaign_sensitivity["fee_proxy_scenario"] == "base")
].copy()

campaign_base.to_csv(
    CSV_DIR / "campaign_simulator_outputs.csv",
    index=False
)
campaign_sensitivity.to_csv(
    CSV_DIR / "campaign_sensitivity_analysis.csv",
    index=False
)

campaign_sensitivity

# 5. Simulated Experiment Analysis

## 5.1 A/B Test Design

Simulated experiment analysis to demonstrate A/B testing logic. The assignment and outcomes below are generated, not observed production experiment data.

In [ ]:
ab_test_design = pd.DataFrame([
    {
        "field": "data_source",
        "definition": "Simulated randomized assignment and outcomes; not production causal evidence."
    },
    {
        "field": "unit_of_randomization",
        "definition": "User"
    },
    {
        "field": "control",
        "definition": "Existing onboarding without a feature prompt"
    },
    {
        "field": "treatment",
        "definition": "Onboarding prompt introducing rate-alert or target-rate order"
    },
    {
        "field": "primary_metric",
        "definition": "Feature adoption within 7 days"
    },
    {
        "field": "secondary_metric",
        "definition": "First exchange within 14 days"
    },
    {
        "field": "guardrail_metric",
        "definition": "Failed transaction or support ticket within 30 days"
    },
    {
        "field": "minimum_practical_effect",
        "definition": "2 percentage-point absolute uplift in the primary metric"
    }
])

ab_test_design.to_csv(
    CSV_DIR / "ab_test_design.csv",
    index=False
)

ab_test_design

### 5.1.1 Simulated Random Assignment and Outcomes

In [ ]:
experiment_users = pd.read_sql_query(
    '''
    SELECT
        user_id,
        acquisition_channel,
        CASE
            WHEN successful_transaction_volume >= 10000000
            THEN 'High value'
            ELSE 'Standard value'
        END AS value_segment
    FROM mart_user_lifecycle
    WHERE onboarding_valid_flag = 1
    ''',
    conn
)

experiment_rng = np.random.default_rng(RANDOM_STATE)

experiment_data = experiment_users.copy()
experiment_data["variant"] = experiment_rng.choice(
    ["control", "treatment"],
    size=len(experiment_data)
)

experiment_data["variant"].value_counts()

In [ ]:
# These probabilities are assumptions for demonstrating experiment analysis
channel_probability_adjustment = (
    experiment_data["acquisition_channel"]
    .map({
        "referral": 0.03,
        "direct": 0.02,
        "organic_search": 0.01,
        "paid_social": -0.01
    })
    .fillna(0)
)

treatment_flag = (
    experiment_data["variant"] == "treatment"
).astype(float)

feature_adoption_probability = np.clip(
    0.21
    + channel_probability_adjustment
    + (0.045 * treatment_flag),
    0.02,
    0.95
)

first_exchange_probability = np.clip(
    0.20
    + channel_probability_adjustment
    + (0.018 * treatment_flag),
    0.02,
    0.95
)

guardrail_probability = np.clip(
    0.075 - (0.004 * treatment_flag),
    0.01,
    0.40
)

experiment_data["feature_adoption_7d"] = (
    experiment_rng.binomial(
        1,
        feature_adoption_probability
    )
)

experiment_data["first_exchange_14d"] = (
    experiment_rng.binomial(
        1,
        first_exchange_probability
    )
)

experiment_data["guardrail_event_30d"] = (
    experiment_rng.binomial(
        1,
        guardrail_probability
    )
)

experiment_data.head()

## 5.2 A/B Test Statistical Evaluation

In [ ]:
def normal_cdf(value):
    return 0.5 * (
        1 + math.erf(value / math.sqrt(2))
    )


def two_proportion_test(control_values, treatment_values):
    control_n = len(control_values)
    treatment_n = len(treatment_values)

    control_rate = control_values.mean()
    treatment_rate = treatment_values.mean()

    pooled_rate = (
        control_values.sum() + treatment_values.sum()
    ) / (control_n + treatment_n)

    pooled_standard_error = math.sqrt(
        max(
            pooled_rate
            * (1 - pooled_rate)
            * ((1 / control_n) + (1 / treatment_n)),
            1e-12
        )
    )

    absolute_uplift = treatment_rate - control_rate
    z_statistic = absolute_uplift / pooled_standard_error
    p_value = 2 * (1 - normal_cdf(abs(z_statistic)))

    difference_standard_error = math.sqrt(
        max(
            (
                control_rate
                * (1 - control_rate)
                / control_n
            )
            + (
                treatment_rate
                * (1 - treatment_rate)
                / treatment_n
            ),
            1e-12
        )
    )

    return {
        "control_n": control_n,
        "treatment_n": treatment_n,
        "control_rate": control_rate,
        "treatment_rate": treatment_rate,
        "absolute_uplift": absolute_uplift,
        "relative_uplift": (
            absolute_uplift / control_rate
            if control_rate > 0
            else np.nan
        ),
        "ci_95_low": (
            absolute_uplift
            - (1.96 * difference_standard_error)
        ),
        "ci_95_high": (
            absolute_uplift
            + (1.96 * difference_standard_error)
        ),
        "z_statistic": z_statistic,
        "p_value": p_value,
        "statistically_significant_5pct": p_value < 0.05
    }

In [ ]:
experiment_metrics = {
    "feature_adoption_7d": "primary",
    "first_exchange_14d": "secondary",
    "guardrail_event_30d": "guardrail"
}

ab_test_result_records = []

for metric_name, metric_role in experiment_metrics.items():
    control_values = experiment_data.loc[
        experiment_data["variant"] == "control",
        metric_name
    ]

    treatment_values = experiment_data.loc[
        experiment_data["variant"] == "treatment",
        metric_name
    ]

    test_result = two_proportion_test(
        control_values,
        treatment_values
    )

    practical_threshold = (
        0.02 if metric_role == "primary" else 0.01
    )

    ab_test_result_records.append({
        "metric": metric_name,
        "metric_role": metric_role,
        **test_result,
        "practically_significant": (
            abs(test_result["absolute_uplift"])
            >= practical_threshold
        ),
        "data_source": "simulated"
    })

ab_test_results = pd.DataFrame(ab_test_result_records)
ab_test_results.to_csv(
    CSV_DIR / "ab_test_results.csv",
    index=False
)

ab_test_results

In [ ]:
ab_segment_records = []

for segment_column in [
    "acquisition_channel",
    "value_segment"
]:
    for segment_value, segment_data in experiment_data.groupby(
        segment_column
    ):
        variant_counts = segment_data["variant"].value_counts()

        if variant_counts.min() < 100:
            continue

        segment_control = segment_data.loc[
            segment_data["variant"] == "control",
            "feature_adoption_7d"
        ]
        segment_treatment = segment_data.loc[
            segment_data["variant"] == "treatment",
            "feature_adoption_7d"
        ]

        segment_result = two_proportion_test(
            segment_control,
            segment_treatment
        )

        ab_segment_records.append({
            "segment_type": segment_column,
            "segment": segment_value,
            "metric": "feature_adoption_7d",
            **segment_result,
            "data_source": "simulated"
        })

ab_test_segment_results = pd.DataFrame(
    ab_segment_records
)

ab_test_segment_results.to_csv(
    CSV_DIR / "ab_test_segment_results.csv",
    index=False
)

ab_test_segment_results.sort_values(
    "absolute_uplift",
    ascending=False
)

The generated treatment group shows higher feature adoption and first exchange in this simulated example. This demonstrates experiment evaluation logic only and must not be presented as proof that the onboarding prompt caused real product uplift.

# 6. Delivery

## 6.1 Dashboard Export Files

In [ ]:
export_inventory = pd.DataFrame({
    "file": [
        path.name
        for path in sorted(CSV_DIR.glob("*.csv"))
    ],
    "size_kb": [
        round(path.stat().st_size / 1024, 1)
        for path in sorted(CSV_DIR.glob("*.csv"))
    ]
})

export_inventory

## 6.2 Limitations and Next Steps

1. Product behavior is synthetic and does not represent actual company performance.
2. Feature adoption comparisons are observational and may reflect user intent.
3. A/B test assignment and outcomes are simulated.
4. Campaign uplift and fee impact are assumption-based proxies, not revenue forecasts.
5. Yahoo Finance provides historical market observations, but the FX signal has not been validated against production operating outcomes.
6. Production next steps include metric validation, experiment instrumentation, probability calibration, fairness review, and drift monitoring.

In [ ]:
conn.close()
print("Notebook completed successfully")